# Multivariate Linear Regression
## Using Snowpark Python and Scikit-Learn
### Overview
This script builds and evaluates a linear model to predict housing prices from multiple features:
- AVERAGE_AREA_INCOME
- AVERAGE_HOME_AGE
- AVERAGE_NUMBER_OF_ROOMS
- AVERAGE_NUMBER_OF_BEDROOMS
- AREA_POPULATION

Remember that the simple regression predicts price from AVERAGE_AREA_INCOME alone. 

We'll see whether this model -- using additional features for prediction -- achieves better accuracy (as measured with the same test data).

Steps:
- Setup
- Load and Explore Data
- Prepare Data for Regression
- Train and Examine Linear Model
- Evaluate the Model
- Examine the Model Visually
### Setup

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.session import Session

config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

* Load configuration and connect to Snowflake

In [ ]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], "rb") as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{"private_key": private_key_bytes}}).create()

### Load and Explore Data
* Define a Snowpark DataFrame on Snowflake table **usa_housing**.

In [ ]:
housingDF = session.table('data_science_db.housing.usa_housing')

* Examine the data.

In [ ]:
housingDF.schema.fields

In [ ]:
housingDF.count()

In [ ]:
housingDF.show(5)

* Look at the correlation between candidate features and the home price. (Note that this cell combines the procedure of a Python loop with the correlation function computed by Snowflake.)

In [ ]:
for feature in ['average_area_income', 'average_home_age', 'average_number_of_rooms',
                'average_number_of_bedrooms', 'area_population']:
    print(feature.ljust(28), housingDF.stat.corr(feature, 'price'))

*Interpretation:* The system shows some correlation between price and the other variables.

### Prepare Data for Regression

In [ ]:
# Drop address column
housingDF = housingDF.drop('address')

# Create train and test sets
(housing_trainDF, housing_testDF) = housingDF.random_split([0.8, 0.2], seed=42)

# Get train and test set sizes
(housing_trainDF.count(), housing_testDF.count())

- Get Pandas DataFrames for training in Scikit-Learn

In [ ]:
train_x_PDF = housing_trainDF.drop('price').toPandas()
train_y_PDF = housing_trainDF.select('price').toPandas()

### Train and Examine Linear Model
- Fit the model to the training data

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(train_x_PDF, train_y_PDF)

- Examine the model parameters

In [ ]:
print('Model intercept: ', lin_reg.intercept_[0])

In [ ]:
import pandas as pd
coeff_PDF = pd.DataFrame(lin_reg.coef_[0], train_x_PDF.columns, columns=['Coefficient'])
coeff_PDF

### Evaluate the Model
- Predict results for the test set

In [ ]:
# Fetch test data as Pandas DataFrames
test_x_PDF = housing_testDF.drop('price').toPandas()
test_y_PDF = housing_testDF.select('price').toPandas()

# Run predictions
predictions = lin_reg.predict(test_x_PDF)

- Display a few predictions

In [ ]:
predictions_PDF = pd.DataFrame(predictions, columns=['prediction'])
test_y_PDF.join(predictions_PDF)[:5]

- Calculate evaluation metrics

Compare these metrics to the metrics from the univariate linear regression model.

In [ ]:
from sklearn import metrics
print('r2:\t', metrics.r2_score(test_y_PDF, predictions))

In [ ]:
import math
print('rmse:\t', round(math.sqrt(metrics.mean_squared_error(test_y_PDF, predictions)), 2))

### Examine the Model Visually

- Scatter plot of predictions vs values 

(A "perfect" and unrealistic model would show all points on a single line with intercept 0 and slope 1.)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(test_y_PDF, predictions)
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.show()

- Distribution of prediction errors

In [ ]:
import seaborn as sns
prediction_errors = test_y_PDF - predictions
prediction_errors = prediction_errors.rename(columns={"PRICE" : "Prediction Error"})
sns.displot(prediction_errors, bins=50)